# Bronze Layer — Yelp Users (Kafka → Iceberg)

Đọc stream từ topic `raw_yelp_users`, ghi raw vào `nessie.bronze.yelp_users`.
Bronze giữ nguyên tất cả records kể cả duplicate — đây là raw landing zone.

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/`.
> Notebook chỉ setup SparkSession, gọi function, và làm phần interactive (monitor/validate).


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.spark_session import get_spark_session

spark = get_spark_session(
    "Yelp_Bronze_Users",
    stop_existing=True
)

print(
    f"✅ SparkSession ready | Endpoint: "
    f"{spark.sparkContext._jsc.hadoopConfiguration().get('fs.s3a.endpoint')}"
)

✅ SparkSession ready | Endpoint: http://minio:9000


26/06/28 13:15:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/28 13:15:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# Đảm bảo boto3 có sẵn (image chưa cài sẵn thì cài lần đầu)
!pip install boto3 --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from src.s3_utils import get_s3_client, ensure_bucket

s3 = get_s3_client()
created = ensure_bucket(s3)
print("✅ Tạo bucket warehouse" if created else "✅ Bucket warehouse đã tồn tại")

✅ Bucket warehouse đã tồn tại


In [4]:
from src.spark_session import stop_active_streams
from src.bronze import create_bronze_table

# Stop bất kỳ stream cũ nào
for name in stop_active_streams(spark):
    print(f"⏹ Stopped stream: {name}")

create_bronze_table(spark)
print("✅ Table nessie.bronze.yelp_users ready")

✅ Table nessie.bronze.yelp_users ready


In [6]:
from src.bronze import start_bronze_stream

print("[*] Khởi chạy Bronze streaming...")
bronze_query = start_bronze_stream(spark, trigger_seconds=15)
print("[+] Bronze stream đang chạy! Topic: raw_yelp_users → nessie.bronze.yelp_users")

[*] Khởi chạy Bronze streaming...
[+] Bronze stream đang chạy! Topic: raw_yelp_users → nessie.bronze.yelp_users


In [ ]:
# Monitor số lượng bản ghi trong Bronze và trạng thái stream mỗi 15 giây
import time

for i in range(1):
    try:
        # 1. Thực hiện query nhưng KHÔNG gọi [0] ngay lập tức
        snapshots = spark.sql("""
            SELECT CAST(summary['total-records'] AS LONG) AS total
            FROM nessie.bronze.yelp_users.snapshots
            ORDER BY committed_at DESC
            LIMIT 1
        """).collect()
        
        # 2. Kiểm tra an toàn: Nếu có snapshot thì lấy, chưa có thì gán bằng 0
        total = snapshots[0]["total"] if len(snapshots) > 0 else 0
        
        # 3. Lấy số lượng luồng đang chạy
        active = len(spark.streams.active)
        print(f"[{i+1:02d}] Bronze records: {total:>10,} | Streams active: {active}")
        
    except Exception as e:
        # Nếu bảng chưa hề tồn tại, nó sẽ bắt lỗi ở đây để không làm chết vòng lặp
        active = len(spark.streams.active)
        print(f"[{i+1:02d}] Đang khởi tạo bảng hoặc chờ batch đầu tiên... (Streams active: {active})")
        
    time.sleep(15)

[01] Bronze records:  1,987,897 | Streams active: 1
